# **Weight Initialization Techniques (Xavier/Glorot & He)**

What not to do in weight initialization:
- Zero initialization
- Non-zero constant initialization
- Random initialization with small weights
- Random initialization with large weights

1. Xavier / Gloat
    - Normal 
    - Uniform
2. He 
    - Normal
    - Uniform

---

# Mathematical Walkthrough of using `Xavier / Glorot`
This document details the exact mathematical steps—from initialization and forward pass to backpropagation and gradient descent—for a single neural network neuron with **two input features**.


## 1. The Setup & Xavier Initialization

Before training begins, we must initialize our weights and bias. Let's look at the shape of our data:
* **Inputs:** Two features ($x_1, x_2$)
* **Outputs:** One predicted value ($\hat{y}$)


### Step 0: Xavier Initialization Math
* **$n_{\text{in}}$ (Fan-in):** $2$ (two input features)
* **$n_{\text{out}}$ (Fan-out):** $1$ (one output neuron)

To initialize the weights ($w_1, w_2$) using Xavier Initialization, we calculate the required variance ($\sigma^2$) for our starting values:

$$\text{Variance } (\sigma^2) = \frac{2}{n_{\text{in}} + n_{\text{out}}} = \frac{2}{2 + 1} = \frac{2}{3} \approx 0.667$$

What we sill do now is we will take the square root of this value 0.667, which will be **0.816**. Whcih means 68% of the weight values will fall btwn -0.816 to +0.816.<br><br>
**How did we find out 68% values will fall btwn these ?**<br>
It is called the Empirical Rule (or the **68-95-99.7 Rule**), and it is a fundamental law of the geometry of a bell curve.

> Higher the number of feature inputs lower will be the variance, and vice-versa. (This inverse relationship is the core secret behind why these weight initializations keep deep neural networks stable.)

The model draws starting weights randomly from a distribution with this variance. Let's assume the starting parameters drawn are:
* **Weight 1 ($w_1$):** $0.4$
* **Weight 2 ($w_2$):** $0.2$
* **Bias ($b$):** $-0.5$

### The Dataset (1 Sample):
* **Input Feature 1 ($x_1$):** $2$
* **Input Feature 2 ($x_2$):** $3$
* **Target ($y$):** $1$
* **Activation Function:** **Sigmoid** $\sigma(z) = \frac{1}{1 + e^{-z}}$


## 2. The Forward Pass

During the forward pass, the model calculates the predicted output ($\hat{y}$ or $a$) and the error (Loss).

### Step A: Calculate the net input ($z$)
We multiply each feature by its respective weight and add the bias:
$$z = (w_1 \cdot x_1) + (w_2 \cdot x_2) + b$$
$$z = (0.4 \cdot 2) + (0.2 \cdot 3) + (-0.5)$$
$$z = 0.8 + 0.6 - 0.5 = \mathbf{0.9}$$

### Step B: Apply the Activation Function to get the prediction ($a$)
We pass $z$ through the Sigmoid activation function:
$$a = \sigma(z) = \frac{1}{1 + e^{-0.9}} \approx \mathbf{0.711}$$

Our model predicts **$0.711$**, but the real target is **$1.0$**.

### Step C: Calculate the Loss ($L$)
We use the Mean Squared Error (MSE) to measure our prediction error:
$$L = (a - y)^2$$
$$L = (0.711 - 1.0)^2 = (-0.289)^2 \approx \mathbf{0.0835}$$


## 3. The Backward Pass (Backpropagation / Chain Rule)

To adjust our weights to make better predictions, we need to know how nudging each weight ($w_1, w_2$) and the bias ($b$) changes the overall Loss ($L$). 

We trace the calculation steps backward using the calculus **Chain Rule**:

$$\text{Weight 1 Gradient: } \frac{\partial L}{\partial w_1} = \frac{\partial L}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial w_1}$$
$$\text{Weight 2 Gradient: } \frac{\partial L}{\partial w_2} = \frac{\partial L}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial w_2}$$
$$\text{Bias Gradient: } \frac{\partial L}{\partial b} = \frac{\partial L}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial b}$$

Let's calculate each of the individual derivatives:

1. **How Loss changes with our prediction ($a$):**
   $$\frac{\partial L}{\partial a} = 2(a - y) = 2(0.711 - 1.0) = -0.578$$

2. **How prediction changes with net input ($z$):**
   $$\frac{\partial a}{\partial z} = a(1 - a) = 0.711 \cdot (1 - 0.711) \approx 0.2055$$

3. **How net input changes with parameters:**
   $$z = w_1 x_1 + w_2 x_2 + b$$
   * Derivative with respect to $w_1$: $\frac{\partial z}{\partial w_1} = x_1 = 2$
   * Derivative with respect to $w_2$: $\frac{\partial z}{\partial w_2} = x_2 = 3$
   * Derivative with respect to bias $b$: $\frac{\partial z}{\partial b} = 1$


## 4. Calculating the Gradients

Now we multiply the chain links together to get our exact gradients:

* **Gradient for Weight 1 ($\frac{\partial L}{\partial w_1}$):**
  $$\frac{\partial L}{\partial w_1} = 2(a - y) \cdot a(1 - a) \cdot x_1$$
  $$\frac{\partial L}{\partial w_1} = -0.578 \cdot 0.2055 \cdot 2 \approx \mathbf{-0.2376}$$

* **Gradient for Weight 2 ($\frac{\partial L}{\partial w_2}$):**
  $$\frac{\partial L}{\partial w_2} = 2(a - y) \cdot a(1 - a) \cdot x_2$$
  $$\frac{\partial L}{\partial w_2} = -0.578 \cdot 0.2055 \cdot 3 \approx \mathbf{-0.3563}$$

* **Gradient for Bias ($\frac{\partial L}{\partial b}$):**
  $$\frac{\partial L}{\partial b} = 2(a - y) \cdot a(1 - a) \cdot 1$$
  $$\frac{\partial L}{\partial b} = -0.578 \cdot 0.2055 \approx \mathbf{-0.1188}$$


## 5. Parameter Update Step (Gradient Descent)

We use Gradient Descent to update our parameters. Let's set a learning rate of **$\eta = 0.1$**:

$$Parameter_{\text{new}} = Parameter_{\text{old}} - \eta \cdot Gradient$$

### Update Weight 1:
$$w_{1, \text{new}} = 0.4 - (0.1 \cdot -0.2376) = 0.4 + 0.02376 = \mathbf{0.42376}$$

### Update Weight 2:
$$w_{2, \text{new}} = 0.2 - (0.1 \cdot -0.3563) = 0.2 + 0.03563 = \mathbf{0.23563}$$

### Update Bias:
$$b_{\text{new}} = -0.5 - (0.1 \cdot -0.1188) = -0.5 + 0.01188 = \mathbf{-0.48812}$$


## Where is this in the Keras Code?

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 1. SETUP (Spawns w1, w2, and b. Initializes them using Xavier/Glorot normal)
model = keras.Sequential([
    layers.Dense(1, activation='sigmoid', input_dim=2, kernel_initializer='glorot_normal')
])

# 2. DEFINE LOSS (MSE)
model.compile(
    loss='mean_squared_error',
    optimizer=keras.optimizers.SGD(learning_rate=0.1)
)

# 3. FORWARD & BACKWARD PASS (Happens automatically during training)
# x = [[2, 3]], y = [[1]]
model.fit(x, y, epochs=1)
```

---

# Mathematical Walkthrough using `He`

This document details the exact mathematical steps—from initialization and forward pass to backpropagation and gradient descent—for a single neural network neuron using **He Initialization** and the **ReLU Activation Function**.


## 1. The Setup & He Initialization

Before training begins, we must initialize our weights and bias. Let's look at the shape of our data:
* **Inputs:** Two features ($x_1, x_2$)
* **Outputs:** One predicted value ($\hat{y}$)

### Step 0: He Initialization Math
* **$n_{\text{in}}$ (Fan-in):** $2$ (two input features)

To initialize the weights ($w_1, w_2$) using He Initialization, we calculate the required variance ($\sigma^2$) for our starting values. Since ReLU shuts off half of the inputs on average, He initialization doubles the variance compared to standard scales:

$$\text{Variance } (\sigma^2) = \frac{2}{n_{\text{in}}} = \frac{2}{2} = 1.0$$

After finding the value of the variance we need to find the **square root** of it, to know spread of the weights.

The model draws starting weights randomly from a distribution with a variance of $1.0$. Let's assume the starting parameters drawn are:
* **Weight 1 ($w_1$):** $0.4$
* **Weight 2 ($w_2$):** $0.2$
* **Bias ($b$):** $-0.5$

### The Dataset (1 Sample):
* **Input Feature 1 ($x_1$):** $2$
* **Input Feature 2 ($x_2$):** $3$
* **Target ($y$):** $1$
* **Activation Function:** **ReLU** $\text{ReLU}(z) = \max(0, z)$


## 2. The Forward Pass

During the forward pass, the model calculates the predicted output ($\hat{y}$ or $a$) and the error (Loss).

### Step A: Calculate the net input ($z$)
We multiply each feature by its respective weight and add the bias:
$$z = (w_1 \cdot x_1) + (w_2 \cdot x_2) + b$$
$$z = (0.4 \cdot 2) + (0.2 \cdot 3) + (-0.5)$$
$$z = 0.8 + 0.6 - 0.5 = \mathbf{0.9}$$

### Step B: Apply the ReLU Activation Function to get the prediction ($a$)
We pass $z$ through the ReLU function:
$$a = \text{ReLU}(0.9) = \max(0, 0.9) = \mathbf{0.9}$$

Our model predicts **$0.9$**, but the real target is **$1.0$**.

### Step C: Calculate the Loss ($L$)
We use the Mean Squared Error (MSE) to measure our prediction error:
$$L = (a - y)^2$$
$$L = (0.9 - 1.0)^2 = (-0.1)^2 = \mathbf{0.01}$$


## 3. The Backward Pass (Backpropagation / Chain Rule)

To adjust our weights to make better predictions, we need to know how nudging each weight ($w_1, w_2$) and the bias ($b$) changes the overall Loss ($L$). 

We trace the calculation steps backward using the calculus **Chain Rule**:

$$\text{Weight 1 Gradient: } \frac{\partial L}{\partial w_1} = \frac{\partial L}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial w_1}$$
$$\text{Weight 2 Gradient: } \frac{\partial L}{\partial w_2} = \frac{\partial L}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial w_2}$$
$$\text{Bias Gradient: } \frac{\partial L}{\partial b} = \frac{\partial L}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial b}$$

Let's calculate each of the individual derivatives:

1. **How Loss changes with our prediction ($a$):**
   $$\frac{\partial L}{\partial a} = 2(a - y) = 2(0.9 - 1.0) = -0.2$$

2. **How prediction changes with net input ($z$) for ReLU:**
   The derivative of ReLU is incredibly simple:
   * If $z > 0$, the derivative is $1$ (the signal passes through completely unchanged).
   * If $z < 0$, the derivative is $0$ (the signal is completely blocked).
   
   Since our $z = 0.9$ (which is greater than 0):
   $$\frac{\partial a}{\partial z} = 1$$

3. **How net input changes with parameters:**
   $$z = w_1 x_1 + w_2 x_2 + b$$
   * Derivative with respect to $w_1$: $\frac{\partial z}{\partial w_1} = x_1 = 2$
   * Derivative with respect to $w_2$: $\frac{\partial z}{\partial w_2} = x_2 = 3$
   * Derivative with respect to bias $b$: $\frac{\partial z}{\partial b} = 1$


## 4. Calculating the Gradients

Now we multiply the chain links together to get our exact gradients:

* **Gradient for Weight 1 ($\frac{\partial L}{\partial w_1}$):**
  $$\frac{\partial L}{\partial w_1} = 2(a - y) \cdot 1 \cdot x_1$$
  $$\frac{\partial L}{\partial w_1} = -0.2 \cdot 1 \cdot 2 = \mathbf{-0.4}$$

* **Gradient for Weight 2 ($\frac{\partial L}{\partial w_2}$):**
  $$\frac{\partial L}{\partial w_2} = 2(a - y) \cdot 1 \cdot x_2$$
  $$\frac{\partial L}{\partial w_2} = -0.2 \cdot 1 \cdot 3 = \mathbf{-0.6}$$

* **Gradient for Bias ($\frac{\partial L}{\partial b}$):**
  $$\frac{\partial L}{\partial b} = 2(a - y) \cdot 1 \cdot 1$$
  $$\frac{\partial L}{\partial b} = -0.2 \cdot 1 \approx \mathbf{-0.2}$$


## 5. Parameter Update Step (Gradient Descent)

We use Gradient Descent to update our parameters. Let's set a learning rate of **$\eta = 0.1$**:

$$Parameter_{\text{new}} = Parameter_{\text{old}} - \eta \cdot Gradient$$

### Update Weight 1:
$$w_{1, \text{new}} = 0.4 - (0.1 \cdot -0.4) = 0.4 + 0.04 = \mathbf{0.44}$$

### Update Weight 2:
$$w_{2, \text{new}} = 0.2 - (0.1 \cdot -0.6) = 0.2 + 0.06 = \mathbf{0.26}$$

### Update Bias:
$$b_{\text{new}} = -0.5 - (0.1 \cdot -0.2) = -0.5 + 0.02 = \mathbf{-0.48}$$


## Where is this in the Keras Code?

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 1. SETUP (Spawns w1, w2, and b. Initializes them using He normal)
model = keras.Sequential([
    layers.Dense(1, activation='relu', input_dim=2, kernel_initializer='he_normal')
])

# 2. DEFINE LOSS (MSE)
model.compile(
    loss='mean_squared_error',
    optimizer=keras.optimizers.SGD(learning_rate=0.1)
)

# 3. FORWARD & BACKWARD PASS (Happens automatically during training)
# x = [[2, 3]], y = [[1]]
model.fit(x, y, epochs=1)
```

---

---


## **Q1: Do we take the square root of the variance to know the x-axis distribution?**

**Yes, absolutely!** 

In statistics, **Variance ($\sigma^2$)** is measured in "squared units" (e.g., squared weights), which is impossible to plot on a standard flat number line. 

By taking the **square root** of the variance, we get the **Standard Deviation ($\sigma$)**:

$$\text{Standard Deviation } (\sigma) = \sqrt{\text{Variance}}$$

Because the Standard Deviation is in the **exact same units** as your weights, it gives you the exact coordinate points on the x-axis (e.g., $\pm 0.816$) where the majority of your weights will be distributed.


## **Q2: Why is this important? Why can't we just use naive random initialization?**

If you initialize weights blindly (like choosing random numbers between $-10$ and $+10$, or setting everything to zero), your network will suffer from one of two training disasters:

### **PROBLEM 1**: The Symmetry Problem (Initializing to Zero)
If you initialize all weights to exactly `0.0`:
* Every neuron in a hidden layer will receive the exact same input signal.
* Every neuron will calculate the exact same activation during the forward pass.
* Every neuron will receive the exact same gradient update during the backward pass.

Because they are perfectly symmetrical, **they will all learn the exact same feature.** Your massive neural network will effectively collapse and behave like a single-neuron model. **Randomness is mandatory to break this symmetry.**

### **PROBLEM 2**: Gradient Explosion or Vanishing (Blind Randomness)
A deep neural network works by multiplying signals across dozens of layers:
* **Weights too large:** Multiplying large numbers layer-after-layer causes activations to blow up to infinity. The gradients explode, causing the model to crash.
* **Weights too small:** Multiplying tiny decimals layer-after-layer causes the signal to shrink to zero. The gradients vanish, and the model stops learning entirely.

**Xavier and He initialization mathematically calculate the exact "Goldilocks" variance** based on the layer size ($n_{\text{in}}$) so that the signal variance remains perfectly stable across all layers.


## **Q3: Is this range of weights applicable after the initialization step?**

**No! It is strictly a ONE-TIME STEP at the absolute beginning of training (Epoch 0).**

Once the training loop begins:
* **The weights are set free:** They are allowed to grow, shrink, or change to whatever values the data demands to minimize the loss.
* **Backpropagation does not enforce the starting boundaries:** If a weight needs to update to `+5.4` to make highly accurate predictions, gradient descent will happily push it there, even if its initial Xavier starting range was limited to $\pm 1.63$.

### The Runner Analogy
Think of weight initialization like placing a runner at the starting line of a race:

```text
  Smart Init (Xavier/He)          Blind Init (Naive Random)
           |                                  |
           V                                  V
   [ Starting Line ]                  [ Lost in the Woods ]
   (Ready to run the track)           (Can't even find the track)
```

* **Xavier/He** places your runner directly on the starting line. Once the starting gun goes off (backpropagation), they can run miles away from the start—but they started at the right spot, so they can easily run the race.
* **Blind Randomness** drops the runner in the middle of a dense jungle miles away from the stadium. Once the race starts, they are so lost they can't even take a step in the right direction.

# **Xavier vs. He Weight Initialization: Pros, Cons, and Use Cases**

This guide breaks down when to use each initialization method, along with their respective advantages and disadvantages in deep learning architectures.


## 1. Xavier (Glorot) Initialization

Xavier initialization is designed for layers using symmetric activation functions that are linear or linear-like around zero.

### **Best Use Cases (When & Where)**
* **Symmetric Activations:** Always pair with **Tanh** or **Sigmoid** activation functions.
* **Recurrent Architectures:** Standard choice for **RNNs** and **LSTMs** since their gating mechanisms rely heavily on Tanh and Sigmoid.
* **Shallow Networks:** Excellent for traditional feedforward networks (MLPs) with 2 to 5 layers.
* **Autoencoders:** Often used in the bottleneck and output reconstruction layers.

### **Pros**
* **Balanced Gradients:** Mathematically balances the variance of both the forward pass (activations) and the backward pass (gradients) by considering both inputs ($n_{\text{in}}$) and outputs ($n_{\text{out}}$).
* **Prevents Saturation:** Keeps initial weights small enough so that Sigmoid and Tanh outputs do not immediately saturate in their flat "dead zones" (where gradients are near 0).

### **Cons**
* **Fails with ReLU:** Does not account for the fact that ReLU turns off 50% of the neurons on average. Using Xavier with ReLU will cause the signal variance to drop by half at each layer, leading to **vanishing gradients** in deep networks.


## 2. He (Kaiming) Initialization

He initialization is designed specifically for rectified (non-symmetric) activation functions that prune negative values.

### **Best Use Cases (When & Where)**
* **Rectified Activations:** Always pair with **ReLU**, **LeakyReLU**, **ELU**, or **PReLU** activation functions.
* **Very Deep Networks:** Essential for deep architectures (e.g., 10 to 100+ layers) like **ResNets**.
* **Computer Vision Models:** The standard default for convolutional neural networks (**CNNs**) which rely heavily on ReLU layers.

### **Pros**
* **Sparsity Compensation:** It mathematically doubles the initialization variance ($\frac{2}{n_{\text{in}}}$ instead of $\frac{1}{n_{\text{in}}}$) to perfectly compensate for ReLU cutting the signal variance in half at every layer.
* **Enables Deep Architectures:** Allows extremely deep neural networks (100+ layers) to converge successfully without early vanishing gradients.

### **Cons**
* **Exploding Gradients with Sigmoid/Tanh:** Because He initialization uses a larger starting variance, pairing it with Sigmoid or Tanh will push the activations into the flat saturated edges of the curves, causing the gradients to vanish immediately.

---

## At-a-Glance Reference Matrix

| Feature | Xavier (Glorot) Initialization | He (Kaiming) Initialization |
| :--- | :--- | :--- |
| **Primary Activations** | `tanh`, `sigmoid` | `relu`, `leaky_relu`, `elu`, `prelu` |
| **Variance Formula** | $\sigma^2 = \frac{2}{n_{\text{in}} + n_{\text{out}}}$ | $\sigma^2 = \frac{2}{n_{\text{in}}}$ |
| **Main Advantage** | Perfect balance for symmetric curves | Prevents signal death in rectified (ReLU) layers |
| **Main Risk** | Signal dies out if used with ReLU | Activations saturate if used with Tanh/Sigmoid |

---

## Quick Code Implementations (Keras)

```python
from tensorflow.keras import layers

# Example 1: Xavier Initialization (for Tanh/Sigmoid)
model.add(layers.Dense(64, activation='tanh', kernel_initializer='glorot_normal'))

# Example 2: He Initialization (for ReLU/ELU/LeakyReLU)
model.add(layers.Dense(64, activation='relu', kernel_initializer='he_normal'))
```

---

Whatever formulas for Xavier and He we learned so far, those were for `normal`, there are `uniform` formulas for both of them also.

# Uniform vs. Normal Initialization: Xavier & He

While **Normal Initialization** draws weights from a bell-shaped curve, **Uniform Initialization** draws weights from a flat, equal-probability range between a negative limit and a positive limit $[-L, +L]$. 


## 1. The Mathematical Formulas

In statistics, the variance ($\sigma^2$) of a continuous uniform distribution between $[-L, +L]$ is calculated as:

$$\text{Variance } (\sigma^2) = \frac{L^2}{3}$$

To match our target variances, we solve for the limit $L$:

$$L = \sqrt{3 \cdot \text{Variance}}$$

### A. Xavier (Glorot) Uniform Formula
* **Target Variance:** $\sigma^2 = \frac{2}{n_{\text{in}} + n_{\text{out}}}$
* **Limit ($L$):** $\sqrt{3 \cdot \frac{2}{n_{\text{in}} + n_{\text{out}}}} = \sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}}$

Weights are drawn with equal probability from the range:

$$\left[ -\sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}}, \,\, +\sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}} \right]$$

### B. He (Kaiming) Uniform Formula
* **Target Variance:** $\sigma^2 = \frac{2}{n_{\text{in}}}$
* **Limit ($L$):** $\sqrt{3 \cdot \frac{2}{n_{\text{in}}}} = \sqrt{\frac{6}{n_{\text{in}}}}$

Weights are drawn with equal probability from the range:

$$\left[ -\sqrt{\frac{6}{n_{\text{in}}}}, \,\, +\sqrt{\frac{6}{n_{\text{in}}}} \right]$$


## 2. How are they different visually?

The difference lies entirely in **how the starting weights are spread out** across the x-axis:

```text
    NORMAL (Bell Curve)                  UNIFORM (Flat Rectangle)
    Most weights are near 0.             Every value has the exact same
    Tails taper off to the sides.        probability of being chosen.

             .                        _____________________
            /     \                      |                     |
          . '     ' .                    |                     |
  _______/_________\_______              |_____________________|
       -Limit     +Limit                 -Limit               +Limit
```

* **Central Tendency:** Under a **Normal** distribution, Keras is highly likely to pick weights very close to `0.0`. Under a **Uniform** distribution, Keras is just as likely to pick a weight near the boundary edges (like $+0.79$) as it is to pick `0.0`.
* **Hard Walls:** A **Uniform** distribution has strict, absolute boundaries. It is physically impossible to draw a weight even a fraction outside of $[-L, +L]$.


## 3. Pros and Cons: Normal vs. Uniform

In practice, both methods perform remarkably well, but they have subtle theoretical and practical differences:

### **Uniform Initialization**
* **Pros:**
  * **No Outliers:** Because of the absolute hard boundaries, there is a zero percent chance of ever generating a wild, extreme outlier weight that could destabilize the network at startup.
  * **Even Symmetry Breaking:** Because weights are spread out completely evenly, they break symmetry very effectively, giving every neuron a highly distinct starting point.
  * **Keras Default:** `glorot_uniform` is the default initializer for most Keras layers because of its extreme stability.
* **Cons:**
  * **Non-Natural:** Real-world data and biological systems rarely distribute features uniformly. It lacks the natural "central clustering" of a normal distribution.

### **Normal Initialization**
* **Pros:**
  * **Biologically & Mathematically Natural:** Most natural phenomena follow a normal distribution (Central Limit Theorem). Having most weights close to zero with only a few larger weights scales beautifully with gradient mathematics.
  * **Excellent for Deep Networks:** `he_normal` is highly favored in very deep networks (like ResNets) because the bell curve shape integrates perfectly with backpropagation flow.
* **Cons:**
  * **Potential Outlier Risk:** Without a truncation safety limit, normal curves can occasionally draw very large weights. (Note: Keras solves this by using a *Truncated Normal* which cuts off the curve at $2$ standard deviations).


## Which one should you choose?

* **Use Xavier/Glorot Uniform (`glorot_uniform`)** as your safe, standard default for **shallow networks**, Tanh, or Sigmoid layers.
* **Use He Normal (`he_normal`)** as your standard default when training **deep convolutional networks** (CNNs) or **dense networks** with **ReLU** activations, as it often provides slightly smoother gradient flow in deep architectures.